This code file takes in a country's economic data and runs a greedy algorithm to maximize its ECI for each budget percentage (5%, 10%, 15%, 20%). Then, the output with all the path increments is saved into a pickle file
#### TO DO:

In [1]:
#TO DO: make sure to create a folder New_result_update in your OWN drive
path = 'drive/My Drive/ECI Research/5.Datasets/New_result_update/' #'drive/My Drive/New_result_update/'

#updated
root_path = "drive/My Drive/ECI Research/5. Datasets/2023 Fall - Data/2019-2022 cleaned/" #'drive/My Drive/ECI Research/Fall 2022/Data/2019-2021 cleaned/'


# TO DO: UPDATE to the main country
main_country = 'Finland'

#### Import packages

Install xlwt:

In [2]:
!pip install xlwt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.0/100.0 kB 1.3 MB/s eta 0:00:00


Install the kaleido package for exporting images

In [3]:
!pip install -U kaleido

#needed to downgrade plotly for kaleido to work; the code seems to run fine with the lower version,
#but let me know if we run into issues
!pip install plotly==5.3.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 8.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
lida 0.0.10 requires fastapi, which is not installed.
lida 0.0.10 requires python-multipart, which is not installed.
lida 0.0.10 requires uvicorn, which is not installed.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.9/23.9 MB 26.5 MB/s eta 0:00:00
  Attempting uninstall: plotly
    Found existing installation: plotly 5.15.0
    Uninstalling plotly-5.15.0:
      Successfully uninstalled plotly-5.15.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
lida 0.0.10 requires fastapi, which is not installed.
lida 0.0.10 requires python-multipart, which is not installed.
lida 0.0.10 requires uvicorn, which is not ins

In [4]:
import io
import os
import csv
import statsmodels.api as sm
import math
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import operator
import xlwt
import tqdm
import sys, os
import multiprocessing
import plotly.graph_objects as go
import plotly.graph_objects as go
import pickle

from plotly.offline import plot
from plotly.subplots import make_subplots
from xlwt import Workbook
from functools import partial
from scipy import linalg as LA
import copy

In [5]:
from functools import partial
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pickle
# seems like pickle has been imported already

In [6]:
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)



#old
# root_path = 'drive/My Drive/ECI Research/Spring 2020/Data/2013-2018 cleaned/'

Mounted at /content/drive/


#### Metrics calculation

MCP:

In [7]:
#### MCP
def compute_mcp(matrix):
  product_list = matrix[0][1::]
  country_list = []
  for i in range(1,len(matrix)):
    country_list.append(matrix[i][0])

  country_sum = []
  for i in range(1,len(matrix)):
    country_sum.append(np.sum(matrix[i][1::]))

  product_sum = []
  for i in range(1,len(matrix[0])):
    product = 0
    for j in range(1,len(matrix)):
        product += matrix[j][i]
    product_sum.append(product)
  total_sum = np.sum(product_sum)

  mcp = []

  for i in range (1,len(matrix)):
    country_row = []
    for j in range(1,len(matrix[i])):
      try:
        percentage = matrix[i][j]/country_sum[i-1]
      except:
        print(country_list[i-1])
      total_percentage = product_sum[j-1]/total_sum
      if percentage > total_percentage:
        country_row.append(1)
      else:
        country_row.append(0)
    mcp.append(country_row)

  return country_list, product_list, mcp, product_sum, country_sum

MCC:

In [8]:
#### MCC
def compute_mcc(mcp):
  mcp2 = mcp
  country_sum = []
  for i in range(0,len(mcp)):
    country_sum.append(np.sum(mcp[i]))

  product_sum = []
  for i in range(0,len(mcp[0])):
    product = 0
    for j in range(0,len(mcp)):
      product += mcp[j][i]
    product_sum.append(product)
  total_sum = np.sum(product_sum)

  mcc = []
  for i in range(0,len(mcp)):
    country_row = []
    for j in range(0,len(mcp)):
      temp_sum = 0
      for k in range(0,len(mcp[0])):
        try:
          value = (mcp[i][k] * mcp2[j][k]/(country_sum[i]*product_sum[k]))
          if np.isinf(value) or np.isnan(value):
            temp_sum += 0
          else:
            temp_sum += value
        except:
          pass
      country_row.append(temp_sum)
    mcc.append(country_row)
  return mcc

MPP:

In [9]:
#### Mpp
def compute_mpp(mcp):
  mcp2 = mcp
  country_sum = []
  for i in range(0,len(mcp)):
    country_sum.append(np.sum(mcp[i]))

  product_sum = []
  for i in range(0,len(mcp[0])):
    product = 0
    for j in range(0,len(mcp)):
      product += mcp[j][i]
    product_sum.append(product)
  total_sum = np.sum(product_sum)

  mpp = []
  for i in range(0,len(mcp[0])):
    product_row = []
    for j in range(0,len(mcp[0])):
      temp_sum = 0
      for k in range(0,len(mcp)):
        try:
          temp_sum += (mcp[k][i] * mcp2[k][j]/(country_sum[k]*product_sum[i]))
        except:
          pass
      product_row.append(temp_sum)
    mpp.append(product_row)
  return mpp


ECI:

In [10]:
#### ECI
def compute_eci(mcc):
  mcc = np.array(mcc)

  e_vals, e_vecs = LA.eig(mcc)
  eigenvectors = e_vecs.tolist()
  eigen = []
  for eigenvector in eigenvectors:
    temp = []
    for col in eigenvector:
      temp.append(col.real)
    eigen.append(temp)

  vector = []
  for i in range(0,len(eigen)):
    vector.append(eigen[i][1])

  eci = []
  if vector[0] < 0:
    for item in vector:
      eci.append((item-np.mean(vector))/np.std(vector))
  else:
    for item in vector:
      eci.append((-item+np.mean(vector))/np.std(vector))
  return eci


PCI:

In [11]:
#### PCI
def compute_pci(mpp):
  mpp = np.array(mpp)
  e_vals, e_vecs = LA.eig(mpp)
  eigenvectors = e_vecs.tolist()
  eigen = []
  for eigenvector in eigenvectors:
    temp = []
    for col in eigenvector:
      temp.append(col.real)
    eigen.append(temp)

  vector = []
  for i in range(0,len(eigen)):
    vector.append(eigen[i][1])

  pci = []
  if vector[len(vector)-2] > 0:
    for item in vector:
      pci.append((item-np.mean(vector))/np.std(vector))
  else:
    for item in vector:
      pci.append((-item+np.mean(vector))/np.std(vector))
  return pci

Proximity:

In [12]:
#### Proximity
def compute_proximity(mcp, product_sum):
  proximity = []
  for i in range(0,len(mcp[0])):
    product_row = []
    for j in range(0,len(mcp[0])):
      temp_sum = 0
      for k in range(0,len(mcp)):
        try:
          temp_sum += mcp[k][i] * mcp[k][j]
        except:
          pass
      p0 = product_sum[i]
      p1 = product_sum[j]
      if p0 < p1:
        large = p1
      else:
        large = p0
      product_row.append(temp_sum/large)
    proximity.append(product_row)
  return proximity

Distance:

In [13]:
#### distance
def compute_distance(mcp, proximity):
  distance = []
  for i in range(0,len(mcp)):
    product_row = []
    for j in range(0,len(mcp[0])):
      sum1 = 0
      for k in range(0,len(mcp[0])):
        if k != j:
          try:
            sum1 += mcp[i][k]*proximity[j][k]
          except:
            pass
      sum2 = 0
      for k in range(0,len(mcp[0])):
        if k != j:
          try:
            sum2 += proximity[j][k]
          except:
            pass
      product_row.append(1-sum1/sum2)
    distance.append(product_row)
  return distance

Opportunity Gain:

In [14]:
#### Opportunity gain

def compute_opportunity_gain(mcp,proximity,pci):
  opportunity_gain = []
  for i in range(0,len(mcp)):
    country_row = []
    for j in range(0,len(mcp[0])):
      gain = 0
      for k in range(0,len(mcp[0])):
        temp = 0
        for l in range(0,len(mcp[0])):
          temp += proximity[l][k]
        gain += (proximity[j][k]/temp)*(1-mcp[i][k])*pci[k]
      gain = gain - (1-distance[i][j]) * pci[j]
      if mcp[i][j] == 1:
        country_row.append(0)
      else:
        country_row.append(gain)
      #country_row.append(gain)
    opportunity_gain.append(country_row)

  opportunity_gain = np.array(opportunity_gain)
  return opportunity_gain

#### marginal oppotunity gain
def compute_marginal_opportunity_gain(mcp, opportunity_gain, country_sum,
                                      product_sum, raw_matrix):
  marginal = []
  total_sum = np.sum(product_sum)
  for i in range(0,len(mcp)):
    country_row = []
    for j in range(0,len(mcp[0])):
      if mcp[i][j] == 1:
        country_row.append(0)
      else:
        export = raw_matrix[i+1][j+1]
        country_export = country_sum[i]
        product_export = product_sum[j]
        export_increase = (product_export * country_export / total_sum) - export
        country_row.append(opportunity_gain[i][j] * export / export_increase)
    marginal.append(country_row)

  marginal = np.array(marginal)
  return marginal

#### Greedy

In [15]:
def two_step_greedy(country, year, num_part, percentage):

    df = pd.read_csv(root_path+f'{year}clean.csv')
    raw_matrix = df.to_numpy() # export data with only values
    matrix = [list(df.columns)] # export data with IDs and values

    for row in raw_matrix:
      if np.min(row[1::]) > 0 :
        matrix.append(list(row))

    product_list = matrix[0][1::]
    country_list = []

    for i in range(1,len(matrix)):         # Search country index
        country_list.append(matrix[i][0])
        if country == 'USA' or country == 'France':
            if country in matrix[i][0]:
                country_index = i
                total_export = np.sum(matrix[i][1::])
                print(f'\nCountry: {country}, index is {i}')
        elif country == matrix[i][0]:
            country_index = i
            total_export = np.sum(matrix[i][1::])
            print(f'\nCountry: {country}, index is {i}')

    budget = (percentage/100) * total_export
    share = budget/num_part
    weapon_index = matrix[0].index('Weapons')
    weapon_value = matrix[country_index][weapon_index]
    print(f'Original total export is {total_export}, budget is {budget}, share is {share}, percentage is {budget/total_export*100}%.')
    mcp, product_sum, country_sum = compute_mcp(matrix)[2:]
    mcc = compute_mcc(mcp)
    eci = compute_eci(mcc)
    mpp = compute_mpp(mcp)
    pci = compute_pci(mpp)
    distances = compute_proximity(mcp, product_sum)

    last_eci = eci[country_index-1]
    print(f'ECI this year is {last_eci}')
    original_vector = np.copy(matrix[country_index][1::])

    total = 0
    improvement_index = []
    improvement_percentage = [0]
    eci_increment = [last_eci]
    improved_matrix = copy.deepcopy(matrix)

    while total + 200 < budget:

        print(f'\nProgress: {round(total/budget, 2)}')

        multiplier = 0

        stop_1 = 0
        stop_2 = 0
        budget_reach = 0

        achieved_eci_1 = 0
        achieved_eci_2 = 0

        total_budget_1 = 0
        total_budget_2 = 0

        last_budget = 0

        while stop_1 == 0 or stop_2 == 0:
            if total + share * multiplier + 200 >= budget:
                budget_reach = 1

            else:
                multiplier += 1

            print(f'Path 1: {stop_1}, Path 2: {stop_2}, multiplier: {multiplier}, keep increasing multiplier.')

            eci_temp = []

            matrix_ = copy.deepcopy(improved_matrix)
            for i in range(1, len(matrix[0])):  # make changes on the matrix with data and IDs, index has to start from 1

                matrix_[country_index][i] += share * multiplier
                country_list, product_list, mcp, product_sum, country_sum = compute_mcp(matrix_)
                mcc = compute_mcc(mcp)

                if i == weapon_index:
                    ratio = (matrix_[country_index][weapon_index] - weapon_value)/weapon_value

                    if ratio > 0.1:
                        eci_temp.append((-1)*np.inf)  # The limit of the increment of weapon is 10%

                    else:

                        eci_temp.append(compute_eci(mcc)[country_index - 1])  # index minus 1 because of the matrix without IDs

                else:
                    eci_temp.append(compute_eci(mcc)[country_index - 1])

                matrix_[country_index][i] -= share * multiplier

            if ((max(eci_temp) > last_eci and total + share * multiplier > last_budget)
                or budget_reach) and stop_1 == 0: # Check whether the eci in path 1 has improved

                stop_1 = 1

                if max(eci_temp) > last_eci:

                    print('Path 1 find increments. Stop path 1.')
                    if eci_temp.count(max(eci_temp)) > 1:

                        indices = np.where(np.array(eci_temp) == np.max(eci_temp))[0]
                        max_pci = np.max(np.array(pci)[indices])
                        index_1 = np.where(np.array(pci) == max_pci)[0][0]

                    elif eci_temp.count(max(eci_temp)) == 1:

                        index_1 = np.argmax(eci_temp)

                elif budget_reach:

                    print(f'Total reached {total + share * multiplier}. Path 1 to the end.')
                    mcp, product_sum, country_sum = compute_mcp(improved_matrix)[2:]
                    proximity = compute_proximity(mcp, product_sum)
                    all_distances = compute_distance(mcp, proximity)
                    distances = all_distances[country_index - 1]
                    index_1 = list(distances).index(min(np.array(distances)[np.array(distances)>0]))
                    compute_distance(mcp, proximity)

                achieved_eci_1 = max(eci_temp)
                total_budget_1 = total + share * multiplier
                last_budget = total_budget_1

            if ((max(eci_temp) > achieved_eci_1 and stop_1 == 1)
                or budget_reach) and stop_2 == 0: # Check whether the eci in path 2 has improved

                stop_2 = 1

                if max(eci_temp) > last_eci:

                    print('Path 2 find increments. Stop path 2.')
                    if eci_temp.count(max(eci_temp)) > 1:

                        indices = np.where(np.array(eci_temp) == np.max(eci_temp))[0]
                        max_pci = np.max(np.array(pci)[indices])
                        index_2 = np.where(np.array(pci) == max_pci)[0][0]

                    elif eci_temp.count(max(eci_temp)) == 1:

                        index_2 = np.argmax(eci_temp)

                elif budget_reach:

                    print(f'Total reached {total + share * multiplier}. Path 2 to the end.')
                    mcp, product_sum, country_sum = compute_mcp(improved_matrix)[2:]
                    proximity = compute_proximity(mcp, product_sum)
                    all_distances = compute_distance(mcp, proximity)
                    distances = all_distances[country_index - 1]
                    index_2 = list(distances).index(min(np.array(distances)[np.array(distances)>0]))

                achieved_eci_2 = max(eci_temp)
                total_budget_2 = total + share * multiplier

        #is this a typo? should be difference instead
        print(f'path 1 differnce:{total_budget_1 - total}')
        print(f'path 2 differnce:{total_budget_2 - total}')
        ratio_1 = (achieved_eci_1 - last_eci)/(total_budget_1 - total)
        ratio_2 = (achieved_eci_2 - last_eci)/(total_budget_2 - total)

        if ratio_1 < ratio_2:

            total = total_budget_2
            print(f'Select path 2, total right now is {total}')
            print(f'Path 2 largest argument is {index_2 + 1}, product is {product_list[index_2]}')
            if achieved_eci_2 > last_eci:

              print(f'last ci is {last_eci}, this eci is {achieved_eci_2}')
              last_eci = achieved_eci_2

            else:
              print(f'last ci is {last_eci}, this eci is {last_eci}')

            improved_matrix[country_index][index_2 + 1] += total_budget_2
            improvement_percentage.append(total_budget_2/budget)
            improvement_index.append(index_2 + 1)

        else:

            total = total_budget_1
            print(f'Select path 1, total right now is {total}')
            print(f'Path 1 largest argument is {index_1 + 1}, product is {product_list[index_1]}')

            if achieved_eci_1 > last_eci:
              print(f'last ci is {last_eci}, this eci is {achieved_eci_1}')
              last_eci = achieved_eci_1

            else:
              print(f'last ci is {last_eci}, this eci is {last_eci}')

            improved_matrix[country_index][index_1 + 1] += total_budget_1
            improvement_percentage.append(total_budget_1/budget)
            improvement_index.append(index_1 + 1)

        eci_increment.append(last_eci)

    optimal_allocation = np.round(np.array(improved_matrix[country_index][1::])\
                                  - original_vector, 2)
    max_eci = last_eci

    return dict(zip(product_list,optimal_allocation)), improvement_percentage, \
            eci_increment, total_export, improvement_index

#This function doesn't seem to be called anywhere; I'm not sure what the use is
def greedy_stablizer(country, year, percentage, num_part):
    num_part = np.linspace(5, 30, num=6)
    result = []
    for part in num_part:
      result.append(two_step_greedy(country, year, percentage, part))
    eci = [i[1] for i in result]
    m = max(eci)
    max_index = [i for i, j in enumerate(eci) if j == m]
    optimal_index = 0
    diversity = sum(1 for i in result[max_index[0]][0].values() if i >= 0)
    for index in max_index:
      if sum(1 for i in result[max_index[index]][0].values() if i >= 0) > diversity:
        optimal_index = index
    return result[optimal_index]

#### Plot

In [16]:
'''
def pie_plot(country, allocations):
    labels = []
    for i in range(len(allocations)):
        allocations[i] = {x:y for x,y in allocations[i].items() if y!=0}
        labels.append(list(allocations[i].keys()))
    specs = [[{'type':'domain'}, {'type':'domain'}, {'type':'domain'}, {'type':'domain'}]]
    fig = make_subplots(rows=1, cols=4, specs = specs)
    fig.add_trace(go.Pie(labels=labels[0], values=np.asarray(list(allocations[0].values()))/np.sum(list(allocations[0].values())),
                         name="5% growth"), 1, 1)
    fig.add_trace(go.Pie(labels=labels[1], values=np.asarray(list(allocations[1].values()))/np.sum(list(allocations[1].values())),
                         name="10% growth"), 1, 2)
    fig.add_trace(go.Pie(labels=labels[2], values=np.asarray(list(allocations[2].values()))/np.sum(list(allocations[2].values())),
                         name="15% growth"), 1, 3)
    fig.add_trace(go.Pie(labels=labels[3], values=np.asarray(list(allocations[3].values()))/np.sum(list(allocations[3].values())),
                         name="20% growth"), 1, 4)

    # Use `hole` to create a donut-like pie chart
    fig.update_traces(hole=.4, hoverinfo="label+percent+name")

    fig.update_layout(
        title_text=f"{country} ECI Growth Optimization Result",
        # Add annotations in the center of the donut pies.
        font=dict(
        family="Times New Roman"),
        annotations=[dict(text='5', x=0.095, y=0.5, font_size=20, showarrow=False),
                     dict(text='10', x=0.37, y=0.5, font_size=20, showarrow=False),
                     dict(text='15', x=0.63, y=0.5, font_size=20, showarrow=False),
                     dict(text='20', x=0.91, y=0.5, font_size=20, showarrow=False)])
    #fig.update_layout(margin=dict(t=50, b=0, l=0, r=0))

    #export plot as png
    fig_name = country + "_allocation_pie.png"
    fig.write_image(fig_name, engine="kaleido")

    fig.show()
'''

def eci_growth_plot(country, eci_increment):
    trace1 = go.Scatter(x = list(range(0, len(eci_increment[0]))),
                         y = eci_increment[0], name="5% growth",
                        line=dict(width=6),marker=dict(size=15))

    trace2 = go.Scatter(x = list(range(0, len(eci_increment[1]))), xaxis="x2",
                         y = eci_increment[1], name="10% growth",
                        line=dict(width=6), marker=dict(size=15))

    trace3 = go.Scatter(x = list(range(0, len(eci_increment[2]))), xaxis="x3",
                         y = eci_increment[2], name="15% growth",
                        line=dict(width=6), marker=dict(size=15))

    trace4 = go.Scatter(x = list(range(0, len(eci_increment[3]))), xaxis="x4",
                         y = eci_increment[3], name="20% growth",
                        line=dict(width=6), marker=dict(size=15))

    data = [trace1, trace2, trace3, trace4]

    layout = go.Layout(
        xaxis=dict(
            domain=[0, 0.25]
        ),
        legend=dict(
            traceorder="reversed"
        ),
        xaxis2=dict(
            domain=[0.25, 0.5]
        ),
        xaxis3=dict(
            domain=[0.5, 0.75]
        ),
        xaxis4=dict(
            domain=[0.75, 1]
        ),
        title_text=f"{country} ECI Growth",
        font=dict(
        family="Times New Roman")

    )
    fig = go.Figure(data=data, layout=layout)
    fig.update_layout(height=400, width=900, title_text=f"{country} ECI Increment", showlegend=True)

    #export plot as png
    fig_name = country + "eci_growth.png"
    fig.write_image(fig_name, engine="kaleido")

    fig.show()

In [17]:
#stop

#Running Greedy Below:

In [18]:
# 5%
# This algo runs for a while (~25min)
country = main_country
year = 2022 #Updated year
num_share = [30, 60, 90, 120]
budget_percentage = [5] # Run for percentages of: [5,10,15,20]

all_result = []
all_allocations = []
all_eci_increments = []

#Running two-step greedy:

for iter, budget in enumerate(budget_percentage):

    result = two_step_greedy(country, year, num_share[iter], budget)
    allocation = result[0]
    eci_increment = result[2]

    all_result.append(result)
    all_allocations.append(allocation)
    all_eci_increments.append(eci_increment)

for j in range(1):
    with open(path + f'{country}_{year}_percentage{budget_percentage[j]}_part{num_share}.p', 'wb') as fp:
          pickle.dump(all_result[j], fp, protocol=pickle.HIGHEST_PROTOCOL)


Country: Finland, index is 41
Original total export is 101561122841.0, budget is 5078056142.05, share is 169268538.06833333, percentage is 5.0%.
ECI this year is -1.304216868015822

Progress: 0.0
Path 1: 0, Path 2: 0, multiplier: 1, keep increasing multiplier.
Path 1: 0, Path 2: 0, multiplier: 2, keep increasing multiplier.
Path 1: 0, Path 2: 0, multiplier: 3, keep increasing multiplier.
Path 1: 0, Path 2: 0, multiplier: 4, keep increasing multiplier.
Path 1: 0, Path 2: 0, multiplier: 5, keep increasing multiplier.
Path 1 find increments. Stop path 1.
Path 1: 1, Path 2: 0, multiplier: 6, keep increasing multiplier.
Path 1: 1, Path 2: 0, multiplier: 7, keep increasing multiplier.
Path 1: 1, Path 2: 0, multiplier: 8, keep increasing multiplier.
Path 1: 1, Path 2: 0, multiplier: 9, keep increasing multiplier.
Path 2 find increments. Stop path 2.
path 1 differnce:846342690.3416667
path 2 differnce:1523416842.615
Select path 1, total right now is 846342690.3416667
Path 1 largest argument i

FileNotFoundError: ignored

In [ ]:
# 10%
# This algo runs for a while (~25min)
country = main_country
year = 2022 #Updated year
num_share = [30, 60, 90, 120]
budget_percentage = [10] # Run for percentages of: [5,10,15,20]

all_result = []
all_allocations = []
all_eci_increments = []

#Running two-step greedy:

for iter, budget in enumerate(budget_percentage):

    result = two_step_greedy(country, year, num_share[iter], budget)
    allocation = result[0]
    eci_increment = result[2]

    all_result.append(result)
    all_allocations.append(allocation)
    all_eci_increments.append(eci_increment)

for j in range(1):
    with open(path + f'{country}_{year}_percentage{budget_percentage[j]}_part{num_share}.p', 'wb') as fp:
          pickle.dump(all_result[j], fp, protocol=pickle.HIGHEST_PROTOCOL)


Country: Finland, index is 41
Original total export is 101561122841.0, budget is 10156112284.1, share is 338537076.13666666, percentage is 10.0%.
ECI this year is -1.304216868015822

Progress: 0.0
Path 1: 0, Path 2: 0, multiplier: 1, keep increasing multiplier.
Path 1: 0, Path 2: 0, multiplier: 2, keep increasing multiplier.
Path 1: 0, Path 2: 0, multiplier: 3, keep increasing multiplier.
Path 1 find increments. Stop path 1.
Path 1: 1, Path 2: 0, multiplier: 4, keep increasing multiplier.
Path 1: 1, Path 2: 0, multiplier: 5, keep increasing multiplier.
Path 2 find increments. Stop path 2.
path 1 differnce:1015611228.41
path 2 differnce:1692685380.6833334
Select path 1, total right now is 1015611228.41
Path 1 largest argument is 1, product is Animals and By-products
last ci is -1.304216868015822, this eci is -0.9729981261909584

Progress: 0.1
Path 1: 0, Path 2: 0, multiplier: 1, keep increasing multiplier.
Path 1: 0, Path 2: 0, multiplier: 2, keep increasing multiplier.
Path 1: 0, Path

FileNotFoundError: ignored

In [ ]:
# 15%
# This algo runs for a while (~20min)

budget_percentage = [15] # Run for percentages of: [5,10,15,20]

all_result = []
all_allocations = []
all_eci_increments = []

#Running two-step greedy:

for iter, budget in enumerate(budget_percentage):

    result = two_step_greedy(country, year, num_share[iter], budget)
    allocation = result[0]
    eci_increment = result[2]

    all_result.append(result)
    all_allocations.append(allocation)
    all_eci_increments.append(eci_increment)

for j in range(1):
    with open(path + f'{country}_{year}_percentage{budget_percentage[j]}_part{num_share}.p', 'wb') as fp:
          pickle.dump(all_result[j], fp, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
# 20%
# This algo runs for a while (~15min)

budget_percentage = [20] # Run for percentages of: [5,10,15,20]

all_result = []
all_allocations = []
all_eci_increments = []

#Running two-step greedy:

for iter, budget in enumerate(budget_percentage):

    result = two_step_greedy(country, year, num_share[iter], budget)
    allocation = result[0]
    eci_increment = result[2]

    all_result.append(result)
    all_allocations.append(allocation)
    all_eci_increments.append(eci_increment)

for j in range(1):
    with open(path + f'{country}_{year}_percentage{budget_percentage[j]}_part{num_share}.p', 'wb') as fp:
          pickle.dump(all_result[j], fp, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
#all_eci_increments